---

# Creating Dummy Data – TEMPLATE

---

In [ ]:
# ==============================================================================
# STEP 1: IMPORT LIBRARIES
# ==============================================================================
# In this step, we import all the necessary tools. We use standard Python 
# libraries for system and random operations, and powerful data science 
# libraries (NumPy and Pandas) to easily create and structure our dummy data.

import os  # Used for operating system interactions (e.g., checking or creating folders)
import random  # Used to generate random choices, numbers, and sample data
import re  # Used for regular expression operations (e.g., text pattern matching)
from datetime import datetime, timedelta  # Used for date arithmetic and generating timestamps

import numpy as np  # Used for efficient numerical operations and handling empty/null values
import pandas as pd  # Used to structure our dummy data into tabular format (DataFrames)

In [ ]:
# ==============================================================================
# STEP 2: CONFIGURATION
# ==============================================================================
# In this step, we define all the global settings and parameters for our dummy
# data generator. This includes the size of the dataset, the structure of the
# survey rounds, file paths, and cleanup rules for sensitive data.

# --- General Generation Settings ---
SEED = 42  # Seed for random number generators to ensure reproducibility of the dummy data
N_ROWS = 500  # Total number of feedback rows to generate across all survey rounds

# --- Survey Round Structure ---
# The dataset will simulate multiple survey rounds over time.
RUN_COUNT = 4              # Number of survey rounds (RunID will range from 1 to RUN_COUNT)
RUN_INTERVAL_DAYS = 14     # Time gap between two consecutive survey rounds (e.g., 14 days = 2 weeks)
RUN_FIELD_DAYS = 5         # The timeframe (in days) during which responses are collected within a single round

# Start date and time for the very first survey round
BASE_DATE = datetime(2026, 1, 1, 8, 0, 0)

# --- Fallbacks & Data Adjustments ---
# The IterationID is usually taken directly from your structural template.
# This fallback is only used if the template does not contain an IterationID.
ITERATION_ID_FALLBACK = "26_01"

# The percentage of empty fields in free-text columns (since not everyone writes a comment)
FREE_TEXT_EMPTY_SHARE = 0.05

# If a free-text comment is classified as ironic/sarcastic, we can shift the 
# quantitative feedback (Likert scale score) slightly to the negative side.
IRONIC_LIKERT_SHIFT = 1

# --- Data Cleaning & Exclusion Lists ---
# List of exact column names that should be excluded (dropped) from the final dataset.
DROP_COLUMNS = [
    "[YOUR COLUMN TO DROP 1]",
    "[YOUR COLUMN TO DROP 2]"
]

# List of prefixes. Any column starting with these words will be dropped.
# Useful if the exact column wording varies slightly between templates.
DROP_COLUMN_PREFIXES = [
    "[YOUR COLUMN START HERE]"
]

# --- File Paths & Output Formats ---
# Path to your local Excel file which will serve as the structural template.
template_file = "./data/[YOUR TEMPLATE FILE HERE].xlsx"

# Local folder where the generated dummy data files will be saved.
output_path = "./dummy"

# File names for the generated CSV and Excel outputs
csv_filename = "synthetic_employee_survey_500_same_structure.csv"
xlsx_filename = "synthetic_employee_survey_500_same_structure.xlsx"

# Set to True to generate an Excel file, or False if you only want a CSV
CREATE_EXCEL = True

# --- Anonymization & Placeholder Settings ---
# To hide the real company name, we replace it with this neutral placeholder in all column headers.
ORG_PLACEHOLDER = "[Organization]"

# Regular expression pattern to detect the company name that should be replaced.
# Ensure this matches the exact name you want to anonymize.
ORG_NAME_PATTERN = re.compile(r"\b[YOUR COMPANYNAME HERE]\b", re.IGNORECASE)

# --- Seed Initialization ---
# Apply the seed to both Python's built-in randomizer and NumPy to ensure identical output on every run.
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ==============================================================================
# STEP 3: COLUMN RENAMING & NORMALIZATION
# ==============================================================================
# In this step, we define how the long survey questions from the template are
# mapped to short, technical column names (e.g., "sat_team" instead of a long question).
#
# Crucial: Renaming is performed ONLY after the data has been fully generated 
# and validated. The generation logic relies on the original question texts to 
# determine values and decide which columns to drop. Renaming early would break this.

# Dictionary mapping original long column titles to their new short titles.
COLUMN_RENAMES = {
    "[YOUR QUESTION/COLUMN TITLE]": "[NEW COLUMN TITLE]"
    # Add all your question-to-shortname mappings here, separated by commas.
}

# Invisible characters (like zero-width spaces or non-breaking spaces) and BOMs
# often slip into Excel column headers. We define them here to clean them up.
ZERO_WIDTH_CHARS = "\u200b\u200c\u200d\ufeff"

def normalize_for_rename(col):
    """
    Normalizes a column header to ensure reliable matching.
    It removes invisible characters, collapses multiple spaces into one,
    trims leading/trailing whitespace, and converts the text to lowercase.
    """
    text = str(col)
    for char in ZERO_WIDTH_CHARS:
        text = text.replace(char, "")
    return " ".join(text.split()).strip().lower()

# Create a lookup dictionary mapping the normalized original names to the short names.
# This is also used later to assign columns to their respective profile dimensions.
COLUMN_RENAME_LOOKUP = {
    normalize_for_rename(original): short
    for original, short in COLUMN_RENAMES.items()
}

def short_column_name(col):
    """
    Returns the short technical name for a given column header if a match exists,
    otherwise returns None.
    """
    return COLUMN_RENAME_LOOKUP.get(normalize_for_rename(col))

def rename_output_columns(df):
    """
    Renames the survey questions in the DataFrame to their technical short names.
    It also prints a summary of successfully renamed columns and lists any
    defined columns that were not found in the actual dataset.
    """
    renamed = {}
    for col in df.columns:
        short = short_column_name(col)
        if short is not None:
            renamed[col] = short
            
    print(f"
Renamed columns: {len(renamed)} out of {len(COLUMN_RENAMES)}")
    
    # Check for defined mappings that did not match any column in the DataFrame
    found = {normalize_for_rename(col) for col in df.columns}
    missing = [
        original for original in COLUMN_RENAMES
        if normalize_for_rename(original) not in found
    ]
    
    if missing:
        print("The following columns were defined for renaming but not found in the dataset:")
        for original in missing:
            print(f"  - {original}")
            
    return df.rename(columns=renamed)

In [ ]:
# ==============================================================================
# STEP 4: SCALE VALUES & LIKERT MAPPING
# ==============================================================================
# In this step, we define the categorical Likert scale options used in the survey.
# We also create a helper function that translates continuous numerical scores
# (which we generate mathematically) back into these human-readable text answers.

# The list of allowed survey responses as they appear in the original template.
# Note: These match the German wording of your template but can be customized.
LIKERT_VALUES = [
    "Stimme voll und ganz zu",
    "Stimme eher zu",
    "Teils/Teils",
    "Stimme eher nicht zu",
    "Stimme gar nicht zu",
    "Kann ich nicht beurteilen"
]

def score_to_likert(score, allow_unknown=True):
    """
    Converts a continuous numerical score into a discrete categorical Likert response.
    Clamps the value between 1 and 5 to ensure it maps to one of our scale points.
    """
    # Introduce a small probability (3.5%) that a respondent selects the "I don't know"
    # option (e.g., 'Kann ich nicht beurteilen'), if allowed for this column.
    if allow_unknown and np.random.random() < 0.035:
        return "Kann ich nicht beurteilen"
    
    # Restrict the numeric score to be strictly between 1.0 and 5.0, then round it
    score = int(round(max(1, min(5, score))))
    
    # Map the rounded integer to the corresponding survey text
    mapping = {
        1: "Stimme gar nicht zu",
        2: "Stimme eher nicht zu",
        3: "Teils/Teils",
        4: "Stimme eher zu",
        5: "Stimme voll und ganz zu"
    }
    
    return mapping[score]

In [ ]:
# ==============================================================================
# STEP 5: SYNTHETIC CATEGORICAL VALUES
# ==============================================================================
# To guarantee strict data privacy, we do not use any real names of departments,
# teams, locations, projects, or organizational codes. Instead, we define lists
# of realistic-sounding but completely fictional categories. These will be 
# randomly assigned to the generated feedback records.

# Fictional business areas/segments
SYNTHETIC_AREAS = [
    'IN-AL (Innovationsfeld Alpha)',
    'ZU-BE (Zukunftssegment Beta)',
    'FO-GA (Fokusbereich Gamma)',
    'TR-DE (Transformationsfeld Delta)',
    'KE-EP (Kernbereich Epsilon)'
]

# Fictional organizational units/hubs
SYNTHETIC_UNITS = [
    'Pionier-Einheit 01',
    'Zukunfts-Hub 02',
    'Fokus-Zelle 03',
    'Inkubator-Einheit 04',
    'Wertstrom-Einheit 05',
    'Spezialisten-Kreis 06',
    'Synergie-Hub 07'
]

# Fictional team names
SYNTHETIC_TEAMS = [
    'Team Vorreiter',
    'Team Impuls',
    'Team Fokus',
    'Team Dynamik',
    'Team Vision',
    'Team Nexus',
    'Team Evolution',
    'Team Synergie'
]

# Fictional context/meeting types where survey feedback is collected
SYNTHETIC_CONTEXTS = [
    'Strategischer Austausch;',
    'Kollaborativer Arbeitsmodus;',
    'Wissensaustausch und Lernen;',
    'Konzeptionelles Arbeiten;',
    'Tägliches operatives Geschäft;',
    'Kreativ- und Ideenfindung;',
    'Planung und Ausrichtung;',
    'Reflektion und Verbesserung;',
    'Kein spezifischer Fokus;'
]

In [ ]:
# ==============================================================================
# STEP 6: FREE-TEXT GENERATION POOLS (THEMES, EMOTIONS & STYLES)
# ==============================================================================
# This section defines the linguistic engine of our dummy generator. 
# To create realistic comments that can be used to test Sentiment Analysis 
# and Clustering algorithms, we model comments across three dimensions:
#
#   1. TOPIC: What the comment is about (useful for Clustering/Topic Modeling).
#   2. EMOTION: The sentiment and arousal level (useful for Sentiment Analysis).
#   3. STYLE: How it is written (length, typos, emojis, capitals, bullet points).
#
# Everything is completely synthetic. No real employee names, locations, 
# or business-critical project names are used.

# ------------------------------------------------------------------------------
# 1. THEMATIC SENTENCE POOLS (TOPICS)
# Each entry is a standalone sentence. They can be dynamically combined 
# with other sentences to form longer, coherent comments.
# ------------------------------------------------------------------------------
TOPICS = {
    "fuehrung": {  # Leadership
        "pos": [
            "Meine Führungskraft nimmt sich wirklich Zeit für uns, das merkt man im Alltag.",
            "Ich fühle mich von meiner Leitung ernst genommen, auch wenn ich mal Kritik äußere.",
            "Unsere Teamleitung hält uns den Rücken frei, das ist viel wert.",
            "Entscheidungen kommen klar und ich muss nicht wochenlang auf Rückmeldung warten.",
            "Mein Chef hat zugegeben, dass er sich geirrt hat, und das hat mich beeindruckt.",
        ],
        "neg": [
            "Führung findet bei uns praktisch nicht statt, jede Entscheidung versandet irgendwo.",
            "Auf eine Rückmeldung von oben warte ich teilweise wochenlang.",
            "Kritik wird angehört, geändert wird danach nichts.",
            "Es wird viel von Vertrauen geredet und trotzdem jedes Detail kontrolliert.",
            "Unsere Leitung wechselt so oft, dass niemand mehr weiß, wer zuständig ist.",
            "Meine Führungskraft hat für alles Zeit, nur nicht für ihr Team.",
        ],
        "amb": [
            "Meine direkte Führungskraft ist gut, weiter oben verliert sich das leider.",
            "Die Absichten der Leitung sind gut, im Alltag kommt davon wenig an.",
            "Manche Entscheidungen sind nachvollziehbar, andere kommen aus dem Nichts.",
        ],
    },
    "meetings": {  # Meetings & Collaboration
        "pos": [
            "Seit wir die Meetings kürzer halten, komme ich endlich wieder zum Arbeiten.",
            "Unsere Retros sind ehrlich und bringen wirklich etwas.",
            "Der wöchentliche Austausch im Team ist für mich der wichtigste Termin der Woche.",
            "Dass wir meetingfreie Freitage haben, war eine der besten Entscheidungen.",
        ],
        "neg": [
            "Ich sitze den halben Tag in Terminen und mache die eigentliche Arbeit abends.",
            "Für jede Kleinigkeit wird ein neuer Termin aufgesetzt, statt einmal kurz zu fragen.",
            "Die Hälfte der Meetings hat weder Agenda noch Ergebnis.",
            "Mein Kalender ist so voll, dass ich keine Stunde am Stück zum Denken finde.",
            "In den großen Runden reden immer dieselben drei Leute.",
        ],
        "amb": [
            "Manche Runden sind wichtig, in Summe sind es aber deutlich zu viele.",
            "Die Formate sind gut gedacht, nur leider viel zu lang.",
        ],
    },
    "workload": {  # Workload & Stress
        "pos": [
            "Die Arbeitsmenge ist gerade gut machbar, das war lange nicht so.",
            "Ich schaffe meine Aufgaben in der regulären Zeit und das tut gut.",
            "Seit die Aufgaben neu verteilt wurden, ist der Druck spürbar geringer.",
        ],
        "neg": [
            "Seit Monaten arbeiten wir am Limit und es kommt trotzdem immer noch etwas obendrauf.",
            "Zwei Stellen sind unbesetzt und die Arbeit verteilt sich einfach auf den Rest.",
            "Ich mache regelmäßig Überstunden, die anscheinend niemanden interessieren.",
            "Abends bin ich so leer, dass ich zu gar nichts mehr komme.",
            "Urlaub nehme ich inzwischen nur noch, um liegengebliebene Arbeit aufzuholen.",
        ],
        "amb": [
            "Stressig ist es schon, es gibt aber auch ruhigere Phasen.",
            "Die Menge wäre okay, wenn nicht ständig etwas dazwischenkäme.",
        ],
    },
    "veraenderung": {  # Change Management / Reorganization
        "pos": [
            "Die neue Struktur ergibt für mich Sinn, endlich sind Zuständigkeiten klar.",
            "Ich finde die Richtung richtig, auch wenn der Weg anstrengend ist.",
            "Dass wir bei der Umstellung mitreden durften, hat viel verändert.",
        ],
        "neg": [
            "Die dritte Umstrukturierung in zwei Jahren: Kaum ist etwas eingespielt, wird alles umgeworfen.",
            "Niemand sagt uns, was die Veränderung konkret für unsere Stellen bedeutet.",
            "Ich habe die Sorge, dass am Ende wieder bei den Leuten gespart wird.",
            "Wir reden über Transformation und schaffen es nicht mal, einen Prozess zu vereinfachen.",
            "Jede Reorganisation kostet uns ein halbes Jahr Produktivität.",
        ],
        "amb": [
            "Die Richtung kann ich verstehen, das Tempo überfordert mich aber.",
            "Vieles wird gerade neu gedacht, ob es besser wird, weiß ich noch nicht.",
        ],
    },
    "arbeitsort": {  # Hybrid Work / Office vs. Home Office
        "pos": [
            "Die flexible Regelung zum mobilen Arbeiten macht meinen Alltag mit Kindern überhaupt erst möglich.",
            "Die Bürotage sind mir wichtig, weil ich die Leute wieder persönlich sehe.",
            "Ich kann mir meine Woche selbst einteilen und das rechne ich uns hoch an.",
        ],
        "neg": [
            "Die neue Präsenzregelung wurde einfach verkündet, gefragt hat uns niemand.",
            "Ich fahre zwei Stunden ins Büro, um dort den ganzen Tag in Videocalls zu sitzen.",
            "Im Büro finde ich oft keinen freien Platz und sitze dann im Flur.",
            "Zu Hause bin ich produktiver, das zählt hier aber offenbar nicht.",
        ],
        "amb": [
            "Ich bin gern im Büro, brauche aber auch meine ruhigen Tage zu Hause.",
            "Hybrid klingt gut, praktisch sitzt dann doch jeder woanders.",
        ],
    },
    "tools": {  # IT Tools & Infrastructure
        "pos": [
            "Das neue Tool spart mir tatsächlich Zeit, da wurde gut zugehört.",
            "Die IT reagiert schnell, wenn wirklich mal etwas klemmt.",
            "Endlich haben wir ein System, in dem ich alles an einer Stelle finde.",
        ],
        "neg": [
            "Unsere Systeme sind so langsam, dass ich jeden Tag Zeit verliere.",
            "Für jede Freigabe brauche ich drei Formulare und zwei Wochen Geduld.",
            "Es werden ständig neue Tools eingeführt, ohne die alten abzuschalten.",
            "Die Daten liegen an vier Stellen und nirgends stimmen sie überein.",
            "Ich pflege Dinge doppelt, weil zwei Systeme nicht miteinander reden.",
        ],
        "amb": [
            "Technisch hat sich viel verbessert, die Prozesse drumherum sind weiter zäh.",
            "Die Software kann viel, nur erklärt sie uns niemand.",
        ],
    },
    "team": {  # Team Spirit & Collaboration
        "pos": [
            "Auf mein Team kann ich mich hundertprozentig verlassen, das trägt mich durch vieles.",
            "Wir lachen viel miteinander und das ist mir echt viel wert.",
            "Wenn jemand ausfällt, fängt das Team das ohne Diskussion auf.",
            "Die Kolleginnen und Kollegen sind der Hauptgrund, warum ich noch hier bin.",
        ],
        "neg": [
            "Im Team ist eine Stimmung entstanden, in der jeder nur noch auf sich schaut.",
            "Konflikte werden bei uns ausgesessen statt angesprochen.",
            "Zwischen den Bereichen wird mehr gegeneinander als miteinander gearbeitet.",
            "Seit den letzten Abgängen fehlt uns die gemeinsame Basis.",
        ],
        "amb": [
            "Menschlich passt es, fachlich ziehen wir aber nicht immer an einem Strang.",
            "Im Kern ist das Team stark, im Detail knirscht es öfter.",
        ],
    },
    "kommunikation": {  # Internal Communication
        "pos": [
            "Ich finde gut, dass inzwischen offener über Fehler gesprochen wird.",
            "Die Infos kommen inzwischen früh genug, um noch etwas beitragen zu können.",
            "Dass die Leitung auch unangenehme Themen benennt, finde ich stark.",
        ],
        "neg": [
            "Wichtige Infos erfahre ich über den Flurfunk statt offiziell.",
            "Es wird viel kommuniziert und wenig wirklich gesagt.",
            "Die Zahlen werden schöngeredet, obwohl alle die Lage kennen.",
            "Entscheidungen erfahren wir erst, wenn sie längst getroffen sind.",
            "Für uns an der Basis übersetzt niemand, was die Strategie konkret bedeutet.",
        ],
        "amb": [
            "Es gibt mehr Informationen als früher, in der Menge verliere ich aber den Überblick.",
            "Kommuniziert wird viel, nur selten zum richtigen Zeitpunkt.",
        ],
    },
    "entwicklung": {  # Career Development & Training
        "pos": [
            "Ich konnte dieses Jahr eine Weiterbildung machen, die mir richtig weitergeholfen hat.",
            "Bei meinen Aufgaben lerne ich viel Neues und das motiviert mich.",
            "Mir wurde eine Aufgabe zugetraut, die ich mir selbst noch nicht zugetraut hätte.",
        ],
        "neg": [
            "Für Weiterbildung ist angeblich Budget da, freigegeben wird am Ende nichts.",
            "Ich mache seit Jahren dasselbe und sehe keine Perspektive.",
            "Stellen werden extern besetzt, obwohl es intern passende Leute gäbe.",
            "Entwicklungsgespräche sind bei uns reine Formsache.",
        ],
        "amb": [
            "Entwicklung ist möglich, man muss aber sehr hartnäckig sein.",
            "Lernen ja, Zeit dafür eher nein.",
        ],
    },
    "wertschaetzung": {  # Recognition & Appreciation
        "pos": [
            "Ein ehrliches Danke von meiner Führungskraft hat mir letztens richtig gutgetan.",
            "Unsere Arbeit wurde zuletzt sichtbar gemacht, das hat das ganze Team gefreut.",
            "Ich merke, dass meine Erfahrung hier zählt.",
        ],
        "neg": [
            "Man hört nur etwas, wenn etwas schiefgeht.",
            "Wir haben das Projekt gerettet und es hat nicht mal jemand erwähnt.",
            "Wertschätzung besteht bei uns aus einer Mail an alle.",
            "Nach fünfzehn Jahren fühlt man sich hier ziemlich austauschbar.",
        ],
        "amb": [
            "Anerkennung gibt es, sie kommt nur selten von denen, von denen ich sie bräuchte.",
            "Im Team wird viel gelobt, von oben kommt dazu nichts.",
        ],
    },
    "verguetung": {  # Compensation & Benefits
        "pos": [
            "Die letzte Gehaltsrunde war fair und nachvollziehbar.",
            "Die Zusatzleistungen hier sind wirklich gut, das vergisst man leicht.",
        ],
        "neg": [
            "Die Gehälter halten mit den Preisen einfach nicht mehr mit.",
            "Neue Kolleginnen verdienen mehr als ich nach acht Jahren.",
            "Über Gehalt wird geredet wie über ein Geheimnis.",
            "Wir sollen Verantwortung übernehmen, bezahlt wird sie aber nicht.",
        ],
        "amb": [
            "Das Gehalt geht in Ordnung, im Verhältnis zur Verantwortung ist es grenzwertig.",
            "Geld ist nicht alles, ganz egal ist es mir aber auch nicht.",
        ],
    },
    "prioritaeten": {  # Strategic Priorities
        "pos": [
            "Ich weiß genau, woran ich arbeiten soll, und das macht vieles leichter.",
            "Endlich wurden Themen auch mal gestrichen statt nur ergänzt.",
        ],
        "neg": [
            "Alles ist Priorität eins, also ist am Ende nichts wirklich wichtig.",
            "Wir liefern schnell statt gut und das fällt uns später auf die Füße.",
            "Projekte werden gestartet und nach drei Monaten wieder eingestampft.",
            "Ich weiß oft nicht, welchen Beitrag meine Arbeit eigentlich leisten soll.",
            "Wir diskutieren monatelang und entscheiden am Ende gar nichts.",
        ],
        "amb": [
            "Die Ziele sind ambitioniert, die Mittel dafür fehlen aber.",
            "Die Strategie leuchtet mir ein, mein Alltag hat damit wenig zu tun.",
        ],
    },
    "einarbeitung": {  # Onboarding
        "pos": [
            "Meine Einarbeitung war gut organisiert, ich bin schnell angekommen.",
            "Mein Buddy hat sich richtig Zeit für mich genommen.",
        ],
        "neg": [
            "Neue Kolleginnen und Kollegen werden ins kalte Wasser geworfen, weil niemand Zeit hat.",
            "Wissen hängt bei uns an einzelnen Personen und sonst nirgends.",
            "In meinen ersten Wochen hat mir niemand gesagt, wofür ich eigentlich zuständig bin.",
        ],
        "amb": [
            "Der Start war holprig, das Team hat es aber aufgefangen.",
        ],
    },
    "gesundheit": {  # Well-being & Health
        "pos": [
            "Ich schaffe es inzwischen wieder, meine Pausen zu machen.",
            "Nach meiner Rückkehr wurde gut auf mein Tempo geachtet.",
            "Dass hier offen über Belastung gesprochen wird, hilft mir sehr.",
        ],
        "neg": [
            "Ich schlafe schlecht und nehme die Arbeit gedanklich mit nach Hause.",
            "In meinem Umfeld sind mehrere Leute ausgefallen und das macht mir Sorgen.",
            "Sonntagabends habe ich regelmäßig ein flaues Gefühl im Magen.",
            "Ich funktioniere nur noch, statt zu arbeiten.",
        ],
        "amb": [
            "Ich achte besser auf mich, der Rahmen dafür bleibt aber eng.",
        ],
    },
    "befragung": {  # Feedback about the survey itself
        "pos": [
            "Danke, dass überhaupt gefragt wird.",
            "Gut, dass diese Befragung anonym ist, sonst hätte ich nichts geschrieben.",
        ],
        "neg": [
            "Ich fülle diese Umfrage seit Jahren aus, passiert ist bisher nichts.",
            "Bitte zeigt auch mal, was aus den Ergebnissen tatsächlich wird.",
        ],
        "amb": [
            "Die Fragen treffen meine Situation nur teilweise.",
        ],
    },
}

# Survey-meta comments are structurally placed at the end of generated texts
# and are therefore excluded from the standard randomized topic selection.
META_TOPIC = "befragung"
TOPIC_KEYS = [key for key in TOPICS if key != META_TOPIC]

# ------------------------------------------------------------------------------
# 2. IRONIC & SARCASTIC SENTENCES (THEMATIC POOLS)
# We use positive wording but mean negative things. Having a dedicated pool 
# prevents ironic statements from defaulting to standard "ambivalent" pools, 
# which keeps the phrasing sharply sarcastic to challenge sentiment models.
# ------------------------------------------------------------------------------
IRONIC_VALENZ = "iro"
IRONIC_SENTENCES = {
    "fuehrung": [
        "Unsere Leitung ist beeindruckend entscheidungsfreudig, solange niemand eine Entscheidung braucht.",
        "Ich schätze die klare Linie von oben, sie wechselt ja nur monatlich.",
        "Toll, wie viel Vertrauen man uns schenkt, kontrolliert wird schließlich nur alles.",
    ],
    "meetings": [
        "Unsere Meetingkultur ist vorbildlich effizient, für jede Entscheidung brauchen wir nur sechs Termine.",
        "Ich liebe unsere Abstimmungsrunden, endlich komme ich mal nicht zum Arbeiten.",
    ],
    "workload": [
        "Die Auslastung ist angenehm entspannt, man merkt kaum, dass man drei Rollen gleichzeitig ausfüllt.",
        "Ein Glück, dass wir so schlank aufgestellt sind, dann fällt Urlaub gar nicht erst auf.",
    ],
    "veraenderung": [
        "Die nächste Reorganisation kommt bestimmt, zum Glück haben wir uns von der letzten schon erholt.",
        "Wir sind wunderbar wandlungsfähig, alle sechs Monate ein neues Zielbild hält jung.",
    ],
    "arbeitsort": [
        "Das Büro ist perfekt ausgestattet, man muss nur um sieben da sein, um einen Platz zu finden.",
        "Unser Hybridmodell funktioniert hervorragend, ich sitze im Büro und alle anderen zu Hause.",
    ],
    "tools": [
        "Unsere Systeme sind hochmodern, sie müssen nur dreimal täglich neu gestartet werden.",
        "Für jedes Problem haben wir ein Tool, für manche gleich vier.",
    ],
    "team": [
        "Im Team herrscht bemerkenswerte Harmonie, Konflikte werden einfach nie angesprochen.",
        "Wir sind ein super Team, das merkt man daran, wie viele gerade gehen.",
    ],
    "kommunikation": [
        "Die Informationslage ist erstklassig, alles Wichtige erfahre ich zuverlässig aus dem Flurfunk.",
        "Transparenz wird bei uns großgeschrieben, deshalb steht auch alles hinterher im Protokoll.",
    ],
    "entwicklung": [
        "Weiterbildung wird großzügig gefördert, man muss sie nur selbst bezahlen und im Urlaub machen.",
        "Die Entwicklungsmöglichkeiten sind riesig, man braucht dafür nur rund acht Jahre Geduld.",
    ],
    "wertschaetzung": [
        "Lob gibt es reichlich, meistens in der Weihnachtsmail an alle.",
        "Man fühlt sich hier sehr gesehen, spätestens wenn etwas schiefgeht.",
    ],
    "verguetung": [
        "Die Bezahlung ist absolut fair, deshalb reden wir auch nie darüber.",
        "Wir werden großzügig entlohnt, in Wertschätzung und Obstkorb.",
    ],
    "prioritaeten": [
        "Wir haben klare Prioritäten, aktuell sind es ungefähr zwölf.",
        "Alles ist wichtig und alles ist dringend, das macht die Planung wunderbar einfach.",
    ],
    "einarbeitung": [
        "Das Onboarding ist hervorragend durchdacht, die Unterlagen findet man nach wenigen Wochen von selbst.",
        "Ich wurde bestens eingearbeitet, man gab mir den Zugang zum Wiki.",
    ],
    "gesundheit": [
        "Gesundheit steht bei uns an erster Stelle, gleich nach den Quartalszahlen.",
        "Für die Balance wird viel getan, es gibt jetzt ein Webinar dazu.",
    ],
    "befragung": [
        "Diese Befragung wird bestimmt wieder sehr wirksam, wie die letzten fümf auch.",
        "Ich freue mich schon auf die Maßnahmen, die daraus folgen werden.",
    ],
}

# Integrate the ironic pool as another valence option inside the main TOPICS dictionary.
# This keeps the pick_sentence function and closing-sentence logic fully operational.
for topic, ironic_sentences in IRONIC_SENTENCES.items():
    TOPICS[topic][IRONIC_VALENZ] = ironic_sentences

# ------------------------------------------------------------------------------
# 3. EMOTION & STYLISTIC MAPPING
# Influenced by Plutchik's and Russell's circumplex models of affect: Valence 
# and Arousal. Valence controls which thematic pool we select from, while 
# Arousal directly dictates the writing style (e.g., emojis, punctuation, capitalization).
# ------------------------------------------------------------------------------
EMOTIONS = {
    "freude":        {"valenz": "pos", "arousal": "hoch"},
    "dankbarkeit":   {"valenz": "pos", "arousal": "niedrig"},
    "stolz":         {"valenz": "pos", "arousal": "hoch"},
    "hoffnung":      {"valenz": "pos", "arousal": "niedrig"},
    "sachlich":      {"valenz": "amb", "arousal": "niedrig"},
    "ambivalenz":    {"valenz": "amb", "arousal": "niedrig"},
    "sarkasmus":     {"valenz": "iro", "arousal": "hoch"},
    "frust":         {"valenz": "neg", "arousal": "hoch"},
    "wut":           {"valenz": "neg", "arousal": "hoch"},
    "enttaeuschung": {"valenz": "neg", "arousal": "niedrig"},
    "angst":         {"valenz": "neg", "arousal": "hoch"},
    "erschoepfung":  {"valenz": "neg", "arousal": "niedrig"},
    "resignation":   {"valenz": "neg", "arousal": "niedrig"},
}

# Probability distribution weights based on the respondent's latent satisfaction.
# These distributions deliberately overlap (e.g., satisfied people can still leave 
# constructive criticism, and unsatisfied people can still praise their teams).
# This prevents the sentiment from being trivially predictable just from Likert scores.
EMOTION_WEIGHTS = {
    "zufrieden": {      # Satisfied
        "freude": 18, "dankbarkeit": 16, "stolz": 8, "hoffnung": 10,
        "sachlich": 12, "ambivalenz": 14, "sarkasmus": 2,
        "frust": 8, "wut": 2, "enttaeuschung": 4, "angst": 2,
        "erschoepfung": 3, "resignation": 1,
    },
    "mittel": {         # Neutral / Average
        "freude": 6, "dankbarkeit": 8, "stolz": 3, "hoffnung": 8,
        "sachlich": 14, "ambivalenz": 20, "sarkasmus": 5,
        "frust": 14, "wut": 4, "enttaeuschung": 8, "angst": 4,
        "erschoepfung": 4, "resignation": 2,
    },
    "unzufrieden": {    # Dissatisfied
        "freude": 2, "dankbarkeit": 4, "stolz": 1, "hoffnung": 3,
        "sachlich": 6, "ambivalenz": 10, "sarkasmus": 8,
        "frust": 20, "wut": 12, "enttaeuschung": 12, "angst": 7,
        "erschoepfung": 10, "resignation": 5,
    },
}

# Dynamic sentence openers categorized by selected emotion
OPENERS = {
    "freude": ["Ehrlich gesagt: ", "Ich muss das mal loswerden: ", "", ""],
    "dankbarkeit": ["Danke an alle, die den Laden am Laufen halten. ", "Vielen Dank dafür: ", "", ""],
    "stolz": ["Ich bin schon ein bisschen stolz auf uns. ", "Wir haben dieses Jahr echt was gerissen. ", ""],
    "hoffnung": ["Ich hoffe wirklich, dass das so bleibt. ", "Vielleicht wird es ja gerade besser. ", ""],
    "sachlich": ["", "", "Aus meiner Sicht: ", "Kurz: ", "Nur eine Anmerkung: "],
    "ambivalenz": ["", "", "Zwiespältig. ", "Ich bin da echt hin- und hergerissen. "],
    "sarkasmus": [
        "Läuft ja alles super hier. ",
        "Danke für die Möglichkeit, das zum dritten Mal aufzuschreiben. ",
        "Na klar, alles bestens. ",
        "Toll gemacht, wirklich. ",
    ],
    "frust": ["Mal ganz ehrlich: ", "Was mich echt nervt: ", "", "Immer dasselbe: "],
    "wut": ["Es reicht langsam. ", "Ich bin richtig sauer. ", "Mir platzt hier gleich der Kragen. "],
    "enttaeuschung": ["Ich bin ehrlich enttäuscht. ", "Schade eigentlich. ", "Ich hatte mehr erwartet. ", ""],
    "angst": ["Ich mache mir Sorgen. ", "Mir ist gerade ziemlich mulmig. ", "Ich weiß nicht, wie es weitergeht. ", ""],
    "erschoepfung": ["Ich bin einfach nur müde. ", "Ehrlich, ich kann nicht mehr. ", "Mir fehlt gerade die Kraft. ", ""],
    "resignation": ["Ich habe aufgegeben, daran noch etwas ändern zu wollen. ", "Ist halt so. ", "Bringt ja eh nichts. ", ""],
}

# Dynamic sentence closers categorized by selected valence
CLOSERS = {
    "pos": [
        "Danke dafür!", "Bitte weiter so.", "Das wollte ich mal loswerden.",
        "Ich komme gern zur Arbeit.", "", "", "",
    ],
    "amb": [
        "Mal sehen, wie es weitergeht.", "Das war's von mir.",
        "Vielleicht bin ich da auch zu kritisch.", "", "", "",
    ],
    "iro": [
        "Aber sicher ändert sich das jetzt.", "Ich bin gespannt.",
        "Wird schon.", "Aber wir bleiben ja optimistisch.", "", "",
    ],
    "neg": [
        "Ich hoffe, das liest überhaupt jemand.", "Bitte ändert das endlich.",
        "Sonst verlieren wir noch mehr gute Leute.",
        "Ich frage mich, was mit diesen Umfragen eigentlich passiert.",
        "So halte ich das nicht mehr lange durch.", "", "", "",
    ],
}

# Connective phrases used to chain multiple sentences together
CONNECTORS_SAME = ["", "", "", "Dazu kommt: ", "Und noch etwas: ", "Außerdem: "]
CONNECTORS_TURN_TO_NEG = ["Aber: ", "Was mich stört: ", "Gleichzeitig gilt aber: ", "Der Haken ist: "]
CONNECTORS_TURN_TO_POS = ["Positiv ist: ", "Gut finde ich dagegen: ", "Trotzdem: ", "Was mich hält: "]

# Categorized emojis based on comment valence
EMOJIS = {
    "pos": ["🙂", ":)", "👍", "😊", "❤️"],
    "amb": ["🤷", "😐", "...", "🙃"],
    "iro": ["🙄", "😏", "🤷"],
    "neg": ["😞", "🙄", "😤", ":-(", "😩"],
}

# Specific eye-roll emojis used as stylistic elements for ironic statements
EYE_ROLL_EMOJIS = ["🙄", "🙄", "😏", "🤡"]

# Valences that will trigger ironic elements (scare quotes & eye rolls) when paired with high arousal
IRONIC_STYLE_VALENZEN = ("amb", "iro")

# Words that will be automatically enclosed in quotation marks (scare quotes) in ironic texts.
# Hand-picked list to avoid accidentally highlighting leading words or non-ironic phrases.
IRONIC_QUOTE_WORDS = [
    "Leitung", "Vertrauen", "Meetingkultur", "Abstimmungsrunden",
    "Auslastung", "Reorganisation", "Zielbild", "Hybridmodell", "Systeme",
    "Tool", "Harmonie", "Informationslage", "Transparenz", "Protokoll",
    "Flurfunk", "Weiterbildung", "Entwicklungsmöglichkeiten",
    "Weihnachtsmail", "Bezahlung", "Wertschätzung", "Obstkorb",
    "Prioritäten", "Onboarding", "Wiki", "Gesundheit", "Quartalszahlen",
    "Balance", "Webinar", "Befragung", "Maßnahmen",
]

In [ ]:
# ==============================================================================
# STEP 6.1: STYLE VARIATION & TEXT ASSEMBLY
# ==============================================================================
# Real survey texts are not clean: typos, uppercase letters, 
# emojis, and missing punctuation are part of it.

import random
import numpy as np

UMLAUT_MAP = {"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"}

def apply_typos(text, n_typos):
    """
    Introduces typos: swapped, missing, or duplicate 
    letters as well as written-out umlauts.
    """
    for _ in range(n_typos):
        mode = random.choice(["swap", "drop", "double", "umlaut"])
        if mode == "umlaut":
            candidates = [ch for ch in text if ch in UMLAUT_MAP]
            if candidates:
                ch = random.choice(candidates)
                text = text.replace(ch, UMLAUT_MAP[ch], 1)
                continue
        
        positions = [i for i, ch in enumerate(text) if ch.isalpha()]
        if len(positions) < 3:
            continue
        i = random.choice(positions[1:-1])
        if mode == "swap":
            text = text[:i] + text[i + 1] + text[i] + text[i + 2:]
        elif mode == "drop":
            text = text[:i] + text[i + 1:]
        else:  # Corrected: Duplicate only the single letter instead of the rest string
            text = text[:i] + text[i] + text[i] + text[i + 1:]
    return text

def quote_a_keyword(text):
    """
    Wraps a buzzword from IRONIC_QUOTE_WORDS in distancing 
    quotation marks. If none occurs in the text, it remains unchanged.
    """
    candidates = [word for word in IRONIC_QUOTE_WORDS if word in text]
    if not candidates:
        return text
    word = random.choice(candidates)
    return text.replace(word, f"„{word}“", 1)

def apply_style(text, emotion):
    """
    Simulates different writing habits based on the emotion.
    """
    valenz = EMOTIONS[emotion]["valenz"]
    arousal = EMOTIONS[emotion]["arousal"]
    
    # High arousal leads to exclamation marks and uppercase letters.
    if arousal == "hoch" and random.random() < 0.35:
        text = text.rstrip(".") + random.choice(["!", "!!", "!!!"])
        if emotion in ("wut", "frust") and random.random() < 0.22:
            words = [w for w in text.split() if len(w) > 5 and w[0].isupper()]
            if words:
                word = random.choice(words)
                text = text.replace(word, word.upper(), 1)
                
    # Some people write entirely in lowercase.
    if random.random() < 0.07:
        text = text.lower()
        
    # Some omit commas.
    if random.random() < 0.05:
        text = text.replace(",", "")
        
    # Thought pause.
    if random.random() < 0.08:
        text = text.replace(". ", "... ", 1)
        
    if random.random() < 0.16:
        text = apply_typos(text, random.randint(1, 2))
        
    # Irony shows in writing via quotation marks around the buzzword and 
    # an eye roll emoji. Both only at high arousal, because dry irony 
    # without exclamation marks would otherwise look overdrawn.
    emoji_added = False
    if valenz in IRONIC_STYLE_VALENZEN and arousal == "hoch":
        if random.random() < 0.45:
            text = quote_a_keyword(text)
        if random.random() < 0.40:
            text = text.rstrip() + " " + random.choice(EYE_ROLL_EMOJIS)
            emoji_added = True
            
    if not emoji_added and random.random() < 0.12:
        text = text.rstrip() + " " + random.choice(EMOJIS[valenz])
        
    return text.strip()


# --- Text Assembly Logic ---

def pick_emotion(overall_score):
    """
    Draws an emotion matching the latent satisfaction, 
    but with deliberate fuzziness.
    """
    if overall_score >= 3.8:
        bucket = "zufrieden"
    elif overall_score <= 2.8:
        bucket = "unzufrieden"
    else:
        bucket = "mittel"
        
    weights = EMOTION_WEIGHTS[bucket]
    emotions = list(weights.keys())
    probabilities = np.array(list(weights.values()), dtype=float)
    probabilities = probabilities / probabilities.sum()
    return str(np.random.choice(emotions, p=probabilities))

def pick_sentence(topic, valenz):
    """
    Fetches a sentence for the topic. If the desired pool is missing, 
    it falls back to the ambivalent one.
    """
    pool = TOPICS[topic].get(valenz) or TOPICS[topic]["amb"]
    return random.choice(pool)

def build_body(emotion, n_topics):
    """
    Assembles the body text from individual topic sentences.
    """
    valenz = EMOTIONS[emotion]["valenz"]
    topics = random.sample(TOPIC_KEYS, k=min(n_topics, len(TOPIC_KEYS)))
    sentences = []
    valenzen = []
    flipped = False
    last_connector = None
    
    for index, topic in enumerate(topics):
        if valenz == IRONIC_VALENZ:
            # Ironic texts remain consistently ironic. A shift to 
            # openly positive or negative would dissolve the irony.
            part_valenz = valenz
        elif valenz == "amb":
            # Ambivalence arises through tone shifts within the text.
            if index == 0:
                part_valenz = random.choice(["pos", "amb"])
            else:
                part_valenz = "neg" if random.random() < 0.7 else "amb"
        elif index > 0 and not flipped and random.random() < 0.15:
            # Even clear texts contain at most one counter-pole.
            part_valenz = "neg" if valenz == "pos" else "pos"
            flipped = True
        else:
            part_valenz = valenz
            
        sentence = pick_sentence(topic, part_valenz)
        
        if index == 0:
            pool = [""]
        elif part_valenz == "neg" and valenzen[-1] != "neg":
            pool = CONNECTORS_TURN_TO_NEG
        elif part_valenz == "pos" and valenzen[-1] != "pos":
            pool = CONNECTORS_TURN_TO_POS
        else:
            pool = CONNECTORS_SAME
            
        # Do not use the same connector twice in a row.
        connector = random.choice(pool)
        if connector and connector == last_connector:
            connector = random.choice(pool)
        last_connector = connector
        
        sentences.append(connector + sentence)
        valenzen.append(part_valenz)
        
    # Some people still append a remark about the survey itself.
    if random.random() < 0.09:
        meta_valenz = valenz if valenz in TOPICS[META_TOPIC] else "amb"
        sentences.append(pick_sentence(META_TOPIC, meta_valenz))
        
    return sentences

def generate_comment_candidate(emotion):
    """
    Generates a synthetic free text for the given emotion. 
    The emotion is deliberately drawn outside so that scale answers 
    of the same row can match it.
    """
    valenz = EMOTIONS[emotion]["valenz"]
    
    # Length distribution: many short, few very long answers.
    variant = random.choices(
        ["kurz", "mittel", "lang", "ausfuehrlich", "stichpunkte"],
        weights=[38, 33, 15, 8, 6],
        k=1
    )[0]
    
    n_topics = {
        "kurz": 1,
        "mittel": 2,
        "lang": 3,
        "ausfuehrlich": random.randint(4, 5),
        "stichpunkte": random.randint(2, 4),
    }[variant]
    
    sentences = build_body(emotion, n_topics)
    
    if variant == "stichpunkte":
        bullet = random.choice(["- ", "• ", "* "])
        text = "\n".join(bullet + sentence for sentence in sentences)
    else:
        use_opener = variant != "kurz" or random.random() < 0.5
        use_closer = variant in ("lang", "ausfuehrlich") or random.random() < 0.25
        opener = random.choice(OPENERS[emotion]).strip() if use_opener else ""
        closer = random.choice(CLOSERS[valenz]) if use_closer else ""
        text = " ".join(part for part in [opener, " ".join(sentences), closer] if part)
        
    return apply_style(text, emotion)

def generate_unique_comment(emotion, used_comments):
    """
    Guarantees unique free texts. Tries 500 times by default, 
    otherwise falls back to text extension logic.
    """
    comment = None
    max_attempts = 500
    
    # We first search for a unique comment
    for _ in range(max_attempts):
        candidate = generate_comment_candidate(emotion)
        if candidate not in used_comments:
            comment = candidate
            break  # Cleanly abort loop, no premature return!
            
    # Fallback for extremely rare cases of duplicates:
    if comment is None:
        comment = generate_comment_candidate(emotion)
        while comment in used_comments:
            comment = comment + " " + pick_sentence(
                random.choice(TOPIC_KEYS),
                random.choice(["pos", "neg", "amb"])
            )
            
    # Only at the very end do we register the comment and return it
    used_comments.add(comment)
    return comment

In [ ]:
# ==============================================================================
# STEP 7: AUTOMATIC COLUMN DETECTION & CLASSIFICATION
# ==============================================================================
# Instead of hardcoding which column is which, this section automatically detects
# and classifies columns based on their headers (metadata and structure) and their
# actual content (using heuristics). This ensures high flexibility across templates.

def normalize_colname(col):
    """
    Standardizes a column name by converting it to lowercase and stripping 
    leading/trailing whitespace for reliable matching.
    """
    return str(col).strip().lower()

# --- Metadata Column Matchers ---
# The following functions identify standard metadata fields based on German 
# and English naming conventions commonly found in survey exports.

def is_id_column(col):
    col_l = normalize_colname(col)
    return col_l == "id"

def is_start_time_column(col):
    col_l = normalize_colname(col)
    return "startzeit" in col_l

def is_finish_time_column(col):
    col_l = normalize_colname(col)
    return "fertigstellungszeit" in col_l

def is_email_column(col):
    col_l = normalize_colname(col)
    return "e-mail" in col_l or "email" in col_l

def is_name_column(col):
    col_l = normalize_colname(col)
    return col_l == "name"

def is_language_column(col):
    col_l = normalize_colname(col)
    return "language" in col_l or "sprache" in col_l

def is_free_text_column(col):
    """
    Detects if a column is a free-text comment field based on typical phrasing.
    """
    col_l = normalize_colname(col)
    return (
        "möchtest" in col_l
        and "teilen" in col_l
    ) or (
        "freitext" in col_l
    )

def is_iteration_column(col):
    col_l = normalize_colname(col)
    return "iterationid" in col_l

def is_run_column(col):
    col_l = normalize_colname(col)
    return "runid" in col_l

def is_department_id_column(col):
    col_l = normalize_colname(col)
    return "abteilungsid" in col_l or "organisationseinheit" in col_l

def is_organization_column(col):
    """
    Detects the column stating the organization (e.g., 'Ich arbeite bei...').
    This column will be filled with the neutral placeholder instead of synthetic data.
    """
    col_l = normalize_colname(col)
    return "ich arbeite bei" in col_l

def is_metadata_column(col):
    """
    Checks if a column is a metadata field that should be processed 
    independently of standard survey questions.
    """
    return any([
        is_id_column(col),
        is_start_time_column(col),
        is_finish_time_column(col),
        is_email_column(col),
        is_name_column(col),
        is_language_column(col),
        is_organization_column(col),
        is_iteration_column(col),
        is_run_column(col),
        is_department_id_column(col)
    ])

def synthetic_choice_for_column(col, synthetic_values):
    """
    Maps structural selection columns to their matching synthetic categories.
    The column header tells us the context, and we return a realistic replacement.
    """
    col_l = normalize_colname(col)
    if "kontext" in col_l or "projekt" in col_l:
        return synthetic_values["context"]
    if "team" in col_l or "gruppe" in col_l:
        return synthetic_values["team"]
    if "einheit" in col_l or "abteilung" in col_l:
        return synthetic_values["unit"]
    if "bereich" in col_l or "organisation" in col_l:
        return synthetic_values["area"]
    return synthetic_values["unit"]

# ------------------------------------------------------------------------------
# DEPARTMENT CODES DETECTION
# Columns representing real department abbreviation codes (e.g., 'AB-CD (Muster)') 
# must NOT be included in the dummy dataset. Since codes differ per template, 
# we detect them dynamically using regex pattern matching.
# ------------------------------------------------------------------------------

# Pattern for hyphenated codes (e.g., 'IN-AL (Innovationsfeld Alpha)' or 'AB-CD')
DEPARTMENT_CODE_PATTERN = re.compile(r"^[A-ZÄÖÜ]+(?:-[A-ZÄÖÜ0-9]+)+$")

# Pattern for short codes with explanatory parentheses (e.g., 'B (Bearbeitung)')
DEPARTMENT_SHORT_CODE_PATTERN = re.compile(r"^[A-ZÄÖÜ0-9]{1,6}$")

# Pattern for trailing abbreviations in parentheses (e.g., '... (ABC)')
DEPARTMENT_ABBREVIATION_PATTERN = re.compile(r"\([A-ZÄÖÜ&/ .-]{2,10}\)$")

def is_department_code_column(col):
    """
    Checks if a column name represents a specific department abbreviation.
    This check is strict to avoid accidentally matching standard question texts
    or core metadata columns.
    """
    if any([
        is_id_column(col),
        is_email_column(col),
        is_iteration_column(col),
        is_run_column(col),
        is_department_id_column(col)
    ]):
        return False
        
    name = str(col).strip()
    code = name.split("(")[0].strip()
    
    if DEPARTMENT_CODE_PATTERN.match(code):
        return True
        
    # Short codes are only treated as department columns if followed by a parenthesis.
    if "(" in name and DEPARTMENT_SHORT_CODE_PATTERN.match(code):
        return True
        
    return bool(DEPARTMENT_ABBREVIATION_PATTERN.search(name))

def is_explicitly_dropped_column(col):
    """
    Checks if a column should be dropped based on the DROP_COLUMNS and
    DROP_COLUMN_PREFIXES settings defined in Step 2.
    """
    col_l = normalize_colname(col)
    if col_l in {normalize_colname(name) for name in DROP_COLUMNS}:
        return True
    return any(
        col_l.startswith(normalize_colname(prefix))
        for prefix in DROP_COLUMN_PREFIXES
    )

# Patterns used to detect duplicate columns generated during exports:
#   - '... Column:2' (standard export numbering)
#   - '... Column.1' (Pandas numbering for duplicate headers)
DUPLICATE_COLUMN_PATTERNS = (
    re.compile(r"^(.*:)\d+$"),
    re.compile(r"^(.*)\.\d+$"),
)

def duplicate_base_name(col):
    """
    Returns the original column name if 'col' is a numbered duplicate, 
    otherwise returns None.
    """
    name = str(col).strip()
    for pattern in DUPLICATE_COLUMN_PATTERNS:
        match = pattern.match(name)
        if match:
            return match.group(1)
    return None

def deduplicate_columns(columns):
    """
    Removes numbered duplicates of survey questions. Only keeps the first, 
    un-numbered instance, provided that the un-numbered version actually 
    exists in the template.
    """
    base_names = {normalize_colname(col) for col in columns}
    kept = []
    for col in columns:
        base = duplicate_base_name(col)
        if base is not None and normalize_colname(base) in base_names:
            continue
        kept.append(col)
    return kept

def select_output_columns(columns):
    """
    Defines the final column selection: retains the original order of the template,
    but filters out department columns, dropped columns, and numbered duplicates.
    """
    columns = [
        col for col in columns
        if not is_department_code_column(col)
        and not is_explicitly_dropped_column(col)
    ]
    return deduplicate_columns(columns)

# ------------------------------------------------------------------------------
# INFERRING COLUMN TYPES FROM VALUES
# Rather than guessing the column type from the header name (which fails easily 
# due to free-text question styling), we analyze the actual data in the template.
# No actual template values are copied into the dummy dataset.
# ------------------------------------------------------------------------------

COLUMN_KIND_LIKERT = "Skalenfrage"
COLUMN_KIND_CHOICE = "Auswahlfeld"
COLUMN_KIND_TEXT = "Freitext"
COLUMN_KIND_EMPTY = "leer"

# Heuristics settings for type detection:
MIN_TEXT_LENGTH = 40  # Minimum average character length to qualify as a free-text field
MIN_TEXT_UNIQUE_SHARE = 0.9  # Minimum ratio of unique values to qualify as free-text (since comments are rarely identical)
MIN_LIKERT_SHARE = 0.8  # Minimum ratio of values that must be valid Likert options to qualify as a Likert scale

def infer_column_kind(values):
    """
    Analyzes values in a template column to automatically determine its type.
    """
    filled = [
        str(value).strip() for value in values
        if pd.notna(value) and str(value).strip() != ""
    ]
    if not filled:
        return COLUMN_KIND_EMPTY
        
    distinct = set(filled)
    likert_share = len(distinct & set(LIKERT_VALUES)) / len(distinct)
    if likert_share >= MIN_LIKERT_SHARE:
        return COLUMN_KIND_LIKERT
        
    unique_share = len(distinct) / len(filled)
    average_length = sum(len(value) for value in filled) / len(filled)
    
    # A column is classified as free-text only if it has highly unique 
    # values and those values are sufficiently long. Everything else defaults to a Choice field.
    if unique_share >= MIN_TEXT_UNIQUE_SHARE and average_length >= MIN_TEXT_LENGTH:
        return COLUMN_KIND_TEXT
        
    return COLUMN_KIND_CHOICE

In [ ]:
# ==============================================================================
# STEP 8: LATENT SYNTHETIC DATA LOGIC (PSYCHOMETRIC PROFILES)
# ==============================================================================
# To make the generated survey responses highly realistic, we use a latent profile
# approach. Each simulated respondent gets a unique, internally correlated
# psychometric profile. These scores are kept strictly internal and are never
# saved in the final export files.

# Dimensions where a high score indicates high satisfaction/positivity.
POSITIVE_PROFILE_KEYS = (
    "orientation", "influence", "social", "learning", "balance"
)

# Dimensions where a high score indicates high strain/negativity.
NEGATIVE_PROFILE_KEYS = ("demand", "ambiguity", "energy_empty")

def profile_overall(profile):
    """
    Calculates overall satisfaction as the mean of all dimensions.
    Flips the negative strain scales (6 - value) so that they align with 
    the positive satisfaction direction.
    """
    values = [profile[key] for key in POSITIVE_PROFILE_KEYS]
    values += [6 - profile[key] for key in NEGATIVE_PROFILE_KEYS]
    return np.mean(values)

def apply_ironic_bias(profile):
    """
    Shifts a latent profile towards the negative spectrum.
    People who use sarcasm or irony in free-text comments typically hold critical
    views. This shift ensures their Likert scale ratings align with the tone of 
    their written comments.
    """
    biased = dict(profile)
    for key in POSITIVE_PROFILE_KEYS:
        biased[key] = max(1, biased[key] - IRONIC_LIKERT_SHIFT)
    for key in NEGATIVE_PROFILE_KEYS:
        biased[key] = min(5, biased[key] + IRONIC_LIKERT_SHIFT)
    biased["overall"] = profile_overall(biased)
    return biased

def generate_latent_profile():
    """
    Generates a correlated psychometric profile for an individual.
    Uses normal distributions and linear relationships to simulate how
    different workplace factors (like workload, social support, and clarity) 
    typically interact.
    """
    orientation = max(1, min(5, round(np.random.normal(3.35, 0.90))))
    influence = max(1, min(5, round(np.random.normal(3.20, 0.85))))
    demand = max(1, min(5, round(np.random.normal(3.15, 1.00))))
    
    # Social support is influenced by orientation and influence
    social = max(
        1,
        min(
            5,
            round(
                0.45 * orientation
                + 0.25 * influence
                + np.random.normal(1.25, 0.80)
            )
        )
    )
    
    # Learning and development is influenced by autonomy and clear direction
    learning = max(
        1,
        min(
            5,
            round(
                0.40 * influence
                + 0.25 * orientation
                + np.random.normal(1.35, 0.75)
            )
        )
    )
    
    # Role ambiguity is driven by workload and lack of direction
    ambiguity = max(
        1,
        min(
            5,
            round(
                0.55 * demand
                + 0.20 * (6 - orientation)
                + np.random.normal(0.60, 0.80)
            )
        )
    )
    
    # Work-life balance is a complex composite of multiple strain and support elements
    balance = max(
        1,
        min(
            5,
            round(
                5.4
                - 0.55 * demand
                - 0.25 * ambiguity
                + 0.25 * orientation
                + 0.20 * social
                + np.random.normal(0, 0.70)
            )
        )
    )
    
    # Exhaustion (energy empty) is mostly driven by demand and role ambiguity
    energy_empty = max(
        1,
        min(
            5,
            round(
                0.65 * demand
                + 0.25 * ambiguity
                + np.random.normal(0.40, 0.65)
            )
        )
    )
    
    profile = {
        "orientation": orientation,
        "influence": influence,
        "demand": demand,
        "social": social,
        "learning": learning,
        "ambiguity": ambiguity,
        "balance": balance,
        "energy_empty": energy_empty
    }
    profile["overall"] = profile_overall(profile)
    return profile

# ------------------------------------------------------------------------------
# MAPPING: SHORT QUESTIONS -> LATENT PROFILE DIMENSIONS
#
# We map questions directly to latent dimensions using their short technical names
# (from COLUMN_RENAMES). This is much more robust than keyword-based matching, 
# which easily breaks when survey wordings change.
#
# Schema: "short_column_name": (dimension, inverted_boolean)
# 'inverted_boolean = True' means that high agreement actually implies a 
# negative state (e.g., agreeing to 'ueberlastung' is a negative strain signal).
# ------------------------------------------------------------------------------
QUESTION_DIMENSIONS = {
    # Orientation: Leadership, communication, strategy, direction
    "info_fuehrung_veraenderung": ("orientation", False),
    "produktgestaltung_aktiv": ("orientation", False),
    "kundenfokus": ("orientation", False),
    "businessentscheidungen_richtung": ("orientation", False),
    "ehrlichkeit_fuehrung": ("orientation", False),
    "feedback_nutzen": ("orientation", False),
    "veraenderung_nachvollziehbar": ("orientation", False),
    "strategie_einfluss": ("orientation", False),
    "kulturentwicklung_richtung": ("orientation", False),
    "offene_kommunikation": ("orientation", False),
    "info_geschaeftslage": ("orientation", False),
    "veraenderung_aktiv": ("orientation", False),
    
    # Influence: Autonomy, decision-making, tools
    "gestaltungsspielraum": ("influence", False),
    "eigenverantwortung_entscheidungen": ("influence", False),
    "einbeziehung_fuehrung": ("influence", False),
    "prozessverbesserung_kontinuierlich": ("influence", False),
    "einfluss_arbeitsmenge": ("influence", False),
    "entscheidungsgeschwindigkeit": ("influence", False),
    "veraenderung_vorantreiben": ("influence", False),
    "tools_technologien": ("influence", False),
    "neue_ideen_ausprobieren": ("influence", False),
    "verantwortung_foerderung": ("influence", False),
    
    # Social: Team spirit, collaboration, psychological safety, recognition
    "team_effizienz": ("social", False),
    "zusammenarbeit_team": ("social", False),
    "psychologische_sicherheit": ("social", False),
    "fehlerkultur_keine_vorwuerfe": ("social", False),
    "hilfe_bitten": ("social", False),
    "lob_anerkennung": ("social", False),
    "meinungsfreiheit": ("social", False),
    "zusammenarbeit_schnittstellen": ("social", False),
    "respekt": ("social", False),
    "effizienz_schnittstellen": ("social", False),
    "teamstimmung": ("social", False),
    
    # Learning: Personal growth and continuous improvement
    "staerken_nutzen": ("learning", False),
    "weiterentwicklung_foerderung": ("learning", False),
    "neues_lernen": ("learning", False),
    "anspruchsvolle_aufgaben": ("learning", False),
    "fehlerkultur_massnahmen": ("learning", False),
    "prozesse_hinterfragen": ("learning", False),
    "faehigkeiten_ausbau": ("learning", False),
    "externe_impulse": ("learning", False),
    
    # Clarity (Ambiguity is a strain scale, but positive responses mean high clarity)
    "erwartungsklarheit": ("ambiguity", False),
    "klare_ziele": ("ambiguity", False),
    "strategieklarheit": ("ambiguity", False),
    "arbeitsinformationen": ("ambiguity", False),
    "zielbeitrag_strategie": ("ambiguity", False),
    
    # Workload, Balance & Strain
    "anforderung_faehigkeit_match": ("demand", False),
    "rahmenbedingungen_change": ("balance", False),
    "work_life_balance": ("balance", False),
    "ueberlastung": ("energy_empty", True),
    
    # Composite/Combined questions
    "beitrag_kundennutzen": (("orientation", "influence", "social"), False),
    "beitrag_wettbewerbsfaehigkeit": (("orientation", "influence", "social"), False),
    "arbeitsfreude": (("social", "learning", "balance"), False),
    "arbeitgeber_empfehlung": ("overall", False),
}

# Standard deviations used to add realistic noise to the responses.
# Prevents respondents mapped to the same dimension from getting identical answers.
DIMENSION_NOISE = 0.45
GENERIC_NOISE = 0.60

def satisfaction_score(profile, dimension):
    """
    Returns the score of a dimension aligned with satisfaction polarity (high = positive).
    Flips strain scales if they are referenced directly.
    """
    if dimension == "overall":
        return profile["overall"]
    value = profile[dimension]
    if dimension in NEGATIVE_PROFILE_KEYS:
        return 6 - value
    return value

def question_dimension(col):
    """
    Returns a tuple containing the (dimension, inverted_boolean) mapped to a column.
    Defaults to (None, False) if no mapping is found.
    """
    return QUESTION_DIMENSIONS.get(short_column_name(col), (None, False))

def generate_likert_for_column(col, profile):
    """
    Generates a realistic Likert value for a column based on its mapped dimension.
    If the question is inverted (e.g., 'ueberlastung'), it flips the scores.
    If no mapping is found, it falls back to a generic score based on overall satisfaction.
    """
    dimension, inverted = question_dimension(col)
    
    if dimension is None:
        return score_to_likert(
            profile["overall"] + np.random.normal(0, GENERIC_NOISE)
        )
        
    if isinstance(dimension, tuple):
        score = np.mean([
            satisfaction_score(profile, key) for key in dimension
        ])
    else:
        score = satisfaction_score(profile, dimension)
        
    if inverted:
        score = 6 - score
        
    return score_to_likert(score + np.random.normal(0, DIMENSION_NOISE))

In [ ]:
# ==============================================================================
# STEP 9: TEMPLATE READING & METADATA EXTRACTION
# ==============================================================================
# This step handles loading the Excel template. It extracts the column headers
# (while automatically anonymizing any company names), checks for iteration IDs,
# and classifies the column types. No actual survey responses are copied.

def anonymize_column_name(col):
    """
    Replaces any occurrence of the real company name in a column header
    with a neutral placeholder to ensure the final dummy file is anonymous.
    """
    if not isinstance(col, str):
        return col
    return ORG_NAME_PATTERN.sub(ORG_PLACEHOLDER, col)

def load_iteration_values(template_df):
    """
    Extracts the IterationID values from the template in their original order.
    If the column is missing or empty, it falls back to the configured fallback value.
    """
    iteration_columns = [
        col for col in template_df.columns
        if is_iteration_column(col)
    ]
    if not iteration_columns:
        print("Note: No IterationID column found in the template.")
        return [ITERATION_ID_FALLBACK]
        
    values = [
        value for value in template_df[iteration_columns[0]].tolist()
        if pd.notna(value) and str(value).strip() != ""
    ]
    if not values:
        print("Note: The IterationID column in the template is empty.")
        return [ITERATION_ID_FALLBACK]
        
    return values

def load_column_kinds(template_df, columns):
    """
    Determines the data type (Likert, Choice, Free Text, Empty) for each column.
    Ensures that any column identified as a free-text question is strictly treated
    as such, even if the template contains only a few short answers.
    """
    column_kinds = {}
    for column, original in zip(columns, template_df.columns):
        if is_free_text_column(column):
            column_kinds[column] = COLUMN_KIND_TEXT
        else:
            column_kinds[column] = infer_column_kind(
                template_df[original].tolist()
            )
    return column_kinds

def load_template(template_path):
    """
    Reads the input Excel file to extract:
    1. The column structure (with anonymized company names).
    2. The IterationID (retained from the template).
    3. The data type for each column based on its values.
    
    The actual survey answers from the template are completely ignored.
    """
    # Read the first sheet of the Excel template as raw object types
    template_df = pd.read_excel(
        template_path,
        sheet_name=0,
        dtype=object
    )
    
    columns = [anonymize_column_name(col) for col in template_df.columns]
    if len(columns) == 0:
        raise ValueError("The template file does not contain any columns.")
        
    iteration_values = load_iteration_values(template_df)
    column_kinds = load_column_kinds(template_df, columns)
    
    return columns, iteration_values, column_kinds

In [ ]:
# ==============================================================================
# STEP 10: SYNTHETIC DATASET GENERATION
# ==============================================================================
# This is the main orchestrator. It builds the synthetic dataset row by row
# following the exact column structure and order of the template.
#
# Crucial: Real department abbreviation columns are automatically excluded,
# and cell values are populated purely based on their inferred column kind 
# (e.g., Likert, Text, Choice) rather than hardcoded column headers.

def generate_dataset_like_template(columns, iteration_values, column_kinds,
                                   n=N_ROWS):
    """
    Generates a synthetic DataFrame that mirrors the template's column layout.
    
    Arguments:
        columns: List of anonymized column headers from the template.
        iteration_values: List of IterationIDs extracted from the template.
        column_kinds: Dict mapping columns to their data types (Likert, Text, etc.).
        n: Total number of rows to generate (defaults to N_ROWS configuration).
    """
    # Filter columns to drop unwanted data and duplicates
    output_columns = select_output_columns(columns)
    
    rows = []
    used_comments = set()  # To keep track of generated comments and ensure uniqueness
    
    for i in range(1, n + 1):
        # 1. Generate an internal psychometric profile for this simulated respondent
        profile = generate_latent_profile()
        
        # 2. Determine the core emotion for this respondent's written feedback.
        # We define this before creating Likert values so both data types match.
        emotion = pick_emotion(profile["overall"])
        
        # 3. Apply negative bias to the scale ratings if the emotion is ironic
        if EMOTIONS[emotion]["valenz"] == IRONIC_VALENZ:
            profile = apply_ironic_bias(profile)
            
        # 4. Chronological survey distribution logic:
        # Distribute the total rows 'n' evenly across the 'RUN_COUNT' survey rounds.
        run_id = (i - 1) * RUN_COUNT // n + 1
        
        # Each round starts at a specific interval. We then add random day 
        # and minute offsets so respondents submit their feedback naturally 
        # within the field duration without overlapping.
        run_start = BASE_DATE + timedelta(
            days=(run_id - 1) * RUN_INTERVAL_DAYS
        )
        start_time = run_start + timedelta(
            days=int(np.random.randint(0, RUN_FIELD_DAYS)),
            minutes=int(np.random.randint(0, 540))  # Distribute across a typical 9-hour window
        )
        finish_time = start_time + timedelta(
            minutes=int(np.random.randint(3, 25))  # Simulate typical completion times (3 to 25 minutes)
        )
        
        # 5. Pick synthetic category replacements for this row
        synthetic_values = {
            "area": random.choice(SYNTHETIC_AREAS),
            "unit": random.choice(SYNTHETIC_UNITS),
            "team": random.choice(SYNTHETIC_TEAMS),
            "context": random.choice(SYNTHETIC_CONTEXTS)
        }
        
        # 6. Populate the row columns based on their detected purpose/type
        row = {}
        for col in output_columns:
            if is_id_column(col):
                row[col] = i
                
            elif is_start_time_column(col):
                row[col] = start_time
                
            elif is_finish_time_column(col):
                row[col] = finish_time
                
            elif is_email_column(col):
                row[col] = "anonymous"
                
            elif is_name_column(col):
                row[col] = "anonymous"
                
            elif is_language_column(col):
                # Standard language fallback
                row[col] = "Deutsch"
                
            elif is_organization_column(col):
                row[col] = ORG_PLACEHOLDER
                
            elif is_iteration_column(col):
                # Retain from template. If multiple values existed, cycle through them in order.
                row[col] = iteration_values[(i - 1) % len(iteration_values)]
                
            elif is_run_column(col):
                row[col] = run_id
                
            elif is_department_id_column(col):
                # Generate a random 8-digit technical organization unit ID
                row[col] = int(np.random.randint(90000000, 99999999))
                
            elif column_kinds.get(col) == COLUMN_KIND_LIKERT:
                row[col] = generate_likert_for_column(col, profile)
                
            elif column_kinds.get(col) == COLUMN_KIND_TEXT:
                # Some respondents do not leave a comment based on config probability
                if np.random.random() < FREE_TEXT_EMPTY_SHARE:
                    row[col] = ""
                else:
                    row[col] = generate_unique_comment(
                        emotion=emotion,
                        used_comments=used_comments
                    )
                    
            elif column_kinds.get(col) == COLUMN_KIND_CHOICE:
                row[col] = synthetic_choice_for_column(col, synthetic_values)
                
            else:
                # The template column is completely empty, so we keep it empty.
                row[col] = ""
                
        rows.append(row)
        
    # Assemble the final tabular dataset
    df = pd.DataFrame(rows, columns=output_columns)
    return df

In [ ]:
# ==============================================================================
# STEP 11: EXPORT & SAVE DATASET
# ==============================================================================
# This step saves our freshly generated dummy dataset. It creates the output 
# folder if it does not exist, cleans up any leftover files from previous runs 
# to prevent outdated data mixes, and exports the data to CSV and Excel format.

def save_dataset(df):
    """
    Saves the synthetic DataFrame as a CSV file and optionally as an Excel file.
    
    Before writing, it proactively deletes existing target files. This is important
    because if you change your template structure and a write operation fails,
    you won't accidentally end up using outdated dummy files from previous runs.
    """
    # Create the local output directory if it doesn't exist yet
    os.makedirs(output_path, exist_ok=True)
    
    # Define absolute file paths using the settings from Step 2
    csv_path = os.path.join(output_path, csv_filename)
    excel_path = os.path.join(output_path, xlsx_filename)
    
    # Proactively remove previous exports to avoid stale file states
    for path in (csv_path, excel_path):
        if os.path.exists(path):
            os.remove(path)
            print(f"Removed outdated output file: {path}")
            
    # Save the dataset as CSV. We use 'utf-8-sig' to ensure that Excel 
    # opens the CSV file with correct UTF-8 character encoding (including German Umlauts).
    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig"
    )
    
    excel_created = False
    
    # Save the dataset as Excel if enabled in Step 2 config
    if CREATE_EXCEL:
        try:
            # We use the openpyxl engine for writing Excel (.xlsx) files
            df.to_excel(
                excel_path,
                index=False,
                engine="openpyxl"
            )
            excel_created = True
        except Exception as exc:
            # Friendly error handling if Excel export fails (most commonly due to a locked file)
            print("
Note: Excel file could not be created.")
            print("The CSV version was saved successfully.")
            print("There is currently no Excel output file available (the old one was deleted).")
            print("Most common reason: The Excel file is currently open in another program.")
            print(f"Technical error message: {exc}")
            
    return csv_path, excel_path, excel_created

In [ ]:
# ==============================================================================
# STEP 12: DATASET VALIDATION
# ==============================================================================
# This function acts as a built-in quality control. It performs rigorous tests 
# to ensure that the generated dataset exactly matches the expected structure, 
# holds no sensitive or duplicate columns, contains completely unique free texts, 
# and respects the timeline configurations without any overlapping runs.

def validate_dataset(df, template_columns, column_kinds):
    """
    Validates the generated synthetic dataset against expected logical rules:
    - Row count matches the configuration (N_ROWS).
    - Column structure and order perfectly match the template (excluding dropped/department columns).
    - No forbidden department, dropped, or internal helper columns exist in the output.
    - No question duplicate columns exist.
    - All non-empty free-text comments are completely unique.
    - Responses lie strictly within their configured survey round timelines.
    """
    print("
" + "=" * 72)
    print("VALIDATION")
    print("=" * 72)
    
    # 1. Determine the expected final columns
    expected_columns = select_output_columns(template_columns)
    dropped_columns = [
        col for col in template_columns
        if col not in expected_columns
    ]
    
    print("
Row count:")
    print(len(df))
    
    print("
Columns in template / in the synthetic file:")
    print(f"{len(template_columns)} / {len(df.columns)}")
    
    print(f"
Excluded/dropped columns ({len(dropped_columns)}):")
    if dropped_columns:
        for col in dropped_columns:
            print(f"  - {col}")
    else:
        print("  None found.")
        
    # Check if the column order and headers match the clean expected list
    same_columns = list(df.columns) == list(expected_columns)
    print("
Column structure as expected:")
    print(same_columns)
    
    # 2. Check for leftover department or explicitly excluded columns
    remaining_department_columns = [
        col for col in df.columns
        if is_department_code_column(col)
        or is_explicitly_dropped_column(col)
    ]
    
    # Check for leftover duplicate/numbered question columns
    remaining_duplicates = [
        col for col in df.columns
        if col not in deduplicate_columns(list(df.columns))
    ]
    
    print("
Remaining department or excluded columns in the file:")
    if remaining_department_columns:
        print(remaining_department_columns)
    else:
        print("None. ✓")
        
    print("
Remaining numbered duplicates of the same question:")
    if remaining_duplicates:
        print(remaining_duplicates)
    else:
        print("None. ✓")
        
    # 3. Check for forbidden internal helper or analysis columns
    forbidden_columns = [
        col for col in df.columns
        if "label" in normalize_colname(col)
        or "score" in normalize_colname(col)
        or "cluster" in normalize_colname(col)
        or "sentiment" in normalize_colname(col)
    ]
    print("
Additional Label/Score/Cluster columns:")
    if forbidden_columns:
        print(forbidden_columns)
    else:
        print("None found.")
        
    # 4. Detailed free-text comment validation
    free_text_columns = [
        col for col in df.columns
        if is_free_text_column(col)
    ]
    
    if free_text_columns:
        text_col = free_text_columns[0]
        print("
Free-text column:")
        print(text_col)
        
        # Empty cells are allowed and expected; exclude them from uniqueness checks
        filled_texts = df[text_col][
            df[text_col].astype(str).str.strip() != ""
        ]
        empty_count = len(df) - len(filled_texts)
        
        print("
Empty free-text fields:")
        print(
            f"{empty_count} out of {len(df)} "
            f"({empty_count / len(df):.1%}, expected "
            f"{FREE_TEXT_EMPTY_SHARE:.0%})"
        )
        
        print("
Number of unique free-text comments:")
        print(filled_texts.nunique())
        
        duplicate_count = filled_texts.duplicated().sum()
        print("
Duplicate comments:")
        print(duplicate_count)
        
        if duplicate_count == 0:
            print("✓ All filled comments are completely unique.")
        else:
            print("⚠ Duplicate comments found.")
            
        print("
Text lengths in characters:")
        print(filled_texts.str.len().describe().round(1).to_string())
    else:
        print("
No free-text column detected.")
        
    # 5. Summary of detected column classifications
    print("
Column types according to template classification:")
    for kind in (
        COLUMN_KIND_LIKERT,
        COLUMN_KIND_CHOICE,
        COLUMN_KIND_TEXT,
        COLUMN_KIND_EMPTY
    ):
        kind_columns = [
            col for col in df.columns
            if not is_metadata_column(col)
            and column_kinds.get(col) == kind
        ]
        print(f"
  {kind}: {len(kind_columns)}")
        
        # List non-Likert columns explicitly to easily spot misclassifications
        if kind != COLUMN_KIND_LIKERT:
            for col in kind_columns:
                print(f"      - {col}")
                
    # Spot questions that are Likert scales but don't have a mapped psychometric profile dimension
    unmapped_likert = [
        col for col in df.columns
        if column_kinds.get(col) == COLUMN_KIND_LIKERT
        and question_dimension(col)[0] is None
    ]
    print("
Likert questions without an assigned profile dimension:")
    if unmapped_likert:
        for col in unmapped_likert:
            print(f"  - {col}")
    else:
        print("None. ✓")
        
    # 6. Survey Round (RunID) and Timeframe Validation
    iteration_columns = [col for col in df.columns if is_iteration_column(col)]
    if iteration_columns:
        print("
IterationID values retained from template:")
        print(df[iteration_columns[0]].value_counts().sort_index().to_string())
        
    run_columns = [col for col in df.columns if is_run_column(col)]
    start_columns = [col for col in df.columns if is_start_time_column(col)]
    runs_out_of_window = []
    
    if run_columns:
        run_col = run_columns[0]
        print("
Rows per Survey Round (RunID):")
        print(df[run_col].value_counts().sort_index().to_string())
        
        if start_columns:
            start_col = start_columns[0]
            print(f"
Timeframes per Run (Interval: {RUN_INTERVAL_DAYS} days):")
            for run_id, group in df.groupby(run_col):
                window_start = BASE_DATE + timedelta(
                    days=(int(run_id) - 1) * RUN_INTERVAL_DAYS
                )
                window_end = window_start + timedelta(days=RUN_FIELD_DAYS)
                print(
                    f"  Run {run_id}: "
                    f"{group[start_col].min()} to {group[start_col].max()}"
                )
                if not group[start_col].between(window_start, window_end).all():
                    runs_out_of_window.append(int(run_id))
                    
            print("
Responses outside of their survey round timeframes:")
            if runs_out_of_window:
                print(runs_out_of_window)
            else:
                print("None. ✓")
        else:
            print("
No RunID column detected.")
            
    # ==========================================================================
    # ASSERTIONS (Hard stops to trigger errors if key logic is broken)
    # ==========================================================================
    assert len(df) == N_ROWS, "The generated file does not contain the exact expected row count."
    assert same_columns, "The generated column structure does not match expectations."
    assert not remaining_department_columns, (
        "Forbidden department or excluded columns were found in the final dataset."
    )
    assert not remaining_duplicates, (
        "Numbered question duplicate columns were found in the final dataset."
    )
    assert not forbidden_columns, "Unwanted helper columns (label, score, cluster, etc.) were generated."
    
    if free_text_columns:
        assert duplicate_count == 0, (
            "The generated free-text comments are not completely unique."
        )
        
    if run_columns:
        assert sorted(int(v) for v in df[run_columns[0]].unique()) == list(
            range(1, RUN_COUNT + 1)
        ), f"The RunID column does not strictly contain the values 1 to {RUN_COUNT}."
        assert not runs_out_of_window, (
            "Some generated submit timestamps fall outside their survey round windows."
        )

In [ ]:
# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================
# This is the entry point of our script. It orchestrates the entire workflow:
# 1. Loads the structural Excel template.
# 2. Generates the synthetic dataset based on the template's structure.
# 3. Validates the generated dataset (crucial: BEFORE renaming, as validation 
#    compares against original question texts).
# 4. Renames long survey questions to short, technical column names.
# 5. Saves the final output as CSV and (optionally) Excel.
# 6. Displays a quick preview of the generated dataset in the terminal.

def main():
    """
    Main orchestrator function that executes the entire dummy data generation pipeline.
    """
    # 1. Read structural metadata and classifications from the Excel template
    template_columns, iteration_values, column_kinds = load_template(
        template_file
    )
    
    # 2. Generate the synthetic survey dataset
    df = generate_dataset_like_template(
        columns=template_columns,
        iteration_values=iteration_values,
        column_kinds=column_kinds,
        n=N_ROWS
    )
    
    # 3. Validate the dataset *before* renaming columns.
    # The validation function relies on matching the original template column names.
    validate_dataset(df, template_columns, column_kinds)
    
    # 4. Rename the final output columns to short technical headers
    df = rename_output_columns(df)
    
    # 5. Save the generated dataset to the configured local folder
    csv_path, excel_path, excel_created = save_dataset(df)
    
    # 6. Print execution summary and a preview to the terminal
    print("
Files successfully created:")
    print(f"  CSV: {csv_path}")
    if excel_created:
        print(f"  Excel: {excel_path}")
        
    print("
Preview of the first 5 rows:")
    print(df.head())

if __name__ == "__main__":
    main()